In [ ]:
#!/usr/bin/env python3
import argparse
import os
import subprocess
import sys
from collections import deque
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple

import json
import requests


# ---------- Config ----------

IGNORED_DIRS = {
    ".git",
    "node_modules",
    "__pycache__",
    ".mypy_cache",
    ".pytest_cache",
    ".venv",
    "venv",
    "dist",
    "build",
    ".idea",
    ".vscode",
    "css",
    "include",
    "vendor",
    "vendors",
    "libs",
    "lib",
    "third_party",
    "third-party",
    "site-packages",
    ".next",
    ".nuxt",
    ".angular",
    ".cache",
    "public",
    "static",
    "target",
    ".gradle",
    ".husky",
    ".pnpm",
    "jspm_packages",
    "bower_components",
    "coverage",
    "htmlcov",
}

# Known file names to ignore (lock files, generated deps, etc.)
IGNORED_FILE_NAMES = {
    "package-lock.json",
    "pnpm-lock.yaml",
    "yarn.lock",
    "poetry.lock",
    "Pipfile.lock",
    "Cargo.lock",
    "composer.lock",
    "Gemfile.lock",
    "go.sum",
    "go.work.sum",
    ".gitignore",
}

# Extensions to ignore completely (lock-like)
IGNORED_FILE_EXTS = {
    ".lock",
}

# Obvious binary / multimedia / big artifacts we don't care about
BINARY_EXTS = {
    # images
    ".png", ".jpg", ".jpeg", ".gif", ".bmp", ".webp", ".svg", ".ico",
    # audio
    ".mp3", ".wav", ".ogg", ".flac", ".m4a",
    # video
    ".mp4", ".mkv", ".avi", ".mov", ".webm",
    # archives & binaries
    ".zip", ".tar", ".gz", ".tgz", ".bz2", ".7z", ".rar",
    ".exe", ".dll", ".so", ".dylib",
    ".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx",
}

# Extensions we treat as "likely text" even without probing
PREF_TEXT_EXTS = {
    ".py", ".go", ".js", ".jsx", ".ts", ".tsx", ".java", ".cs", ".php", ".rb",
    ".rs", ".c", ".cc", ".cpp", ".h", ".hpp",
    ".lua", ".sh", ".bash", ".zsh", ".ps1",
    ".tf", ".tfvars",
    ".yaml", ".yml", ".json", ".toml", ".ini", ".cfg",
    ".md", ".rst", ".txt",
    ".html", ".htm", ".css", ".scss", ".less",
    ".svelte", ".vue",
}

DEFAULT_MODEL = "openai/gpt-oss-120b"
TOGETHER_URL = "https://api.together.xyz/v1/chat/completions"


# ---------- Repo / FS helpers ----------

def run_git_clone(repo_url: str, dest_dir: Path) -> None:
    if dest_dir.exists():
        print(f"[INFO] Destination {dest_dir} already exists, skipping clone.", file=sys.stderr)
        return
    print(f"[INFO] Cloning {repo_url} into {dest_dir} ...", file=sys.stderr)
    subprocess.run(["git", "clone", repo_url, str(dest_dir)], check=True)


def should_ignore_file(path: Path) -> bool:
    name = path.name
    ext = path.suffix.lower()

    if name in IGNORED_FILE_NAMES:
        return True
    if ext in IGNORED_FILE_EXTS:
        return True
    # ignore obvious minified bundles
    if name.endswith(".min.js") or name.endswith(".min.css"):
        return True

    return False


def is_probably_text_file(path: Path, max_sample_bytes: int = 2048) -> bool:
    """
    Heuristic to skip obvious binaries; prefer text-like extensions,
    otherwise try to decode a small chunk as UTF-8.
    """
    if should_ignore_file(path):
        return False

    ext = path.suffix.lower()

    if ext in BINARY_EXTS:
        return False

    if ext in PREF_TEXT_EXTS:
        return True

    try:
        with path.open("rb") as f:
            chunk = f.read(max_sample_bytes)
        if not chunk:
            return True
        if b"\x00" in chunk:
            return False
        try:
            chunk.decode("utf-8")
            return True
        except UnicodeDecodeError:
            return False
    except OSError:
        return False


def read_file_text(path: Path) -> str:
    try:
        with path.open("r", encoding="utf-8", errors="replace") as f:
            return f.read()
    except OSError as e:
        print(f"[WARN] Failed to read {path}: {e}", file=sys.stderr)
        return ""


def bfs_directories(root: Path) -> Iterable[Path]:
    """
    Breadth-first traversal over directories starting at root.
    """
    queue = deque([root])
    while queue:
        current = queue.popleft()
        yield current

        try:
            entries = sorted(current.iterdir(), key=lambda p: p.name)
        except OSError as e:
            print(f"[WARN] Cannot list {current}: {e}", file=sys.stderr)
            continue

        for entry in entries:
            if entry.is_dir():
                if entry.name in IGNORED_DIRS:
                    continue
                queue.append(entry)


def collect_files_in_dir(directory: Path) -> List[Path]:
    try:
        entries = sorted(directory.iterdir(), key=lambda p: p.name)
    except OSError as e:
        print(f"[WARN] Cannot list files in {directory}: {e}", file=sys.stderr)
        return []
    return [p for p in entries if p.is_file()]


# ---------- Chunking logic (lines + char limit) ----------

def build_chunks_for_dir(
    files: List[Path],
    root: Path,
    processed_files: Set[Path],
    max_lines_per_chunk: int,
    max_file_size_bytes: int,
    max_chars_per_chunk: int,
) -> Iterable[List[Tuple[Path, str]]]:
    """
    Group files in a directory into chunks respecting:
      - max_lines_per_chunk (sum of lines in chunk)
      - max_chars_per_chunk (sum of characters in chunk)

    Additional rules:
      - If an individual file's size in bytes > max_file_size_bytes -> skipped.
      - If an individual file's text length > max_chars_per_chunk -> skipped.
      - If line_count >= max_lines_per_chunk but char length <= max_chars_per_chunk
        -> file is sent alone as its own chunk.
      - Otherwise, add neighbors until either lines or chars limit would be exceeded.
    """
    i = 0
    n = len(files)

    while i < n:
        path = files[i]

        if path in processed_files:
            i += 1
            continue

        if should_ignore_file(path):
            print(f"[INFO] Skipping ignored file {path}", file=sys.stderr)
            processed_files.add(path)
            i += 1
            continue

        try:
            size = path.stat().st_size
        except OSError:
            size = 0

        if size > max_file_size_bytes:
            print(f"[INFO] Skipping big file {path} ({size} bytes)", file=sys.stderr)
            processed_files.add(path)
            i += 1
            continue

        if not is_probably_text_file(path):
            print(f"[INFO] Skipping non-text file {path}", file=sys.stderr)
            processed_files.add(path)
            i += 1
            continue

        text = read_file_text(path)
        if not text.strip():
            processed_files.add(path)
            i += 1
            continue

        line_count = text.count("\n") + 1
        char_count = len(text)

        # If this file alone breaks 0.75 char limit, just exclude it.
        if char_count > max_chars_per_chunk * 0.75:
            print(
                f"[INFO] Skipping file {path} with {char_count} chars (> max_chars_per_chunk={max_chars_per_chunk})",
                file=sys.stderr,
            )
            processed_files.add(path)
            i += 1
            continue

        chunk: List[Tuple[Path, str]] = []
        total_lines = 0
        total_chars = 0

        # Case 1: this file is "big" wrt line count but still within char limit -> send alone
        if line_count >= max_lines_per_chunk:
            print(
                f"[INFO] File {path} has {line_count} lines (>= {max_lines_per_chunk}), sending alone.",
                file=sys.stderr,
            )
            chunk.append((path, text))
            processed_files.add(path)
            i += 1
        else:
            # Case 2: start chunk with this file and try to add neighbors
            chunk.append((path, text))
            processed_files.add(path)
            total_lines += line_count
            total_chars += char_count
            i += 1

            j = i
            while j < n and total_lines < max_lines_per_chunk and total_chars < max_chars_per_chunk:
                neighbor = files[j]

                if neighbor in processed_files:
                    j += 1
                    continue

                if should_ignore_file(neighbor):
                    print(f"[INFO] Skipping ignored file {neighbor}", file=sys.stderr)
                    processed_files.add(neighbor)
                    j += 1
                    continue

                try:
                    size_n = neighbor.stat().st_size
                except OSError:
                    size_n = 0

                if size_n > max_file_size_bytes or not is_probably_text_file(neighbor):
                    processed_files.add(neighbor)
                    j += 1
                    continue

                neighbor_text = read_file_text(neighbor)
                if not neighbor_text.strip():
                    processed_files.add(neighbor)
                    j += 1
                    continue

                neighbor_lines = neighbor_text.count("\n") + 1
                neighbor_chars = len(neighbor_text)

                # Skip neighbor if it alone exceeds char limit
                if neighbor_chars > max_chars_per_chunk:
                    print(
                        f"[INFO] Skipping file {neighbor} with {neighbor_chars} chars (> max_chars_per_chunk={max_chars_per_chunk})",
                        file=sys.stderr,
                    )
                    processed_files.add(neighbor)
                    j += 1
                    continue

                # If adding it would exceed either limit, stop growing this chunk
                if total_lines + neighbor_lines > max_lines_per_chunk or total_chars + neighbor_chars > max_chars_per_chunk:
                    break

                chunk.append((neighbor, neighbor_text))
                processed_files.add(neighbor)
                total_lines += neighbor_lines
                total_chars += neighbor_chars
                j += 1

            i = j

        if chunk:
            yield chunk


# ---------- Tool analysis helpers ----------

def load_tool_analysis_struct(path: Optional[Path]) -> Optional[Dict]:
    """
    Expect JSON of the form:
    {
      "files": {
        "relative/path.py": {
          "linter": [...],
          "security": [...],
          "tests": [...]
        },
        ...
      }
    }
    """
    if not path:
        return None
    if not path.exists():
        print(f"[WARN] tool-analysis-path {path} does not exist; ignoring.", file=sys.stderr)
        return None
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if not isinstance(data, dict) or "files" not in data:
            print("[WARN] tool analysis JSON has unexpected format; expected top-level 'files' key.", file=sys.stderr)
        return data
    except Exception as e:
        print(f"[WARN] Failed to parse tool analysis JSON from {path}: {e}", file=sys.stderr)
        return None


def build_tool_analysis_for_chunk(
    chunk_files: List[Tuple[Path, str]],
    root: Path,
    tool_analysis_struct: Optional[Dict],
) -> str:
    """
    Given tool_analysis_struct in the { "files": { path: { ... } } } format,
    return a compact text section that includes only entries for files in this chunk.
    """
    if not tool_analysis_struct or "files" not in tool_analysis_struct:
        return ""

    files_map = tool_analysis_struct.get("files", {})
    parts: List[str] = []

    for fpath, _content in chunk_files:
        rel = fpath.relative_to(root).as_posix()
        info = files_map.get(rel)
        if not info:
            continue

        parts.append(f'<TOOL_ANALYSIS_FOR_FILE path="{rel}">')
        parts.append(json.dumps(info, indent=2))
        parts.append("</TOOL_ANALYSIS_FOR_FILE>")

    return "\n\n".join(parts)


# ---------- Prompt building (SiMAL) ----------

def build_file_blocks(chunk_files: List[Tuple[Path, str]], root: Path) -> List[str]:
    blocks: List[str] = []
    for fpath, content in chunk_files:
        rel = fpath.relative_to(root).as_posix()
        block = (
            f'<FILE path="{rel}">\n'
            f"{content}\n"
            f"</FILE>"
        )
        blocks.append(block)
    return blocks


def get_simal_instructions() -> str:
    return """You are a helper that builds and updates SiMAL schemas describing software systems.

Your ONLY output must be a valid SiMAL schema (without any comments, explanations, or extra text).

SiMAL = System Modeling and Annotation Language. It is a compact, human-readable DSL for describing:
- system structure and services,
- runtime/deployment configuration,
- APIs and data models,
- internal components (databases, caches, structs, methods, etc.),
- LLM feedback and issues found by analysis tools,
- Connections and relationships between system elements.

You will receive:
- (Optional) existing SiMAL schema for the system;
- (Optional) tool analysis results (linters, security scanners, static analysis, tests, etc.);
- One or more source files from a single repository, wrapped in tags:
  <FILE path="...">
  ...file content...
  </FILE>

Your task:
- If there is NO existing schema: create a new complete SiMAL schema for this repository, based on the provided files.
- If there IS an existing schema: update it using the new information from the current files and tool outputs.
- You may ADD, MODIFY, or DELETE schema elements, but ONLY when justified by the current input files or tool analysis.
- If some information is unknown or not present in the code, leave it out instead of guessing.

IMPORTANT CONSTRAINTS:
- Output ONLY a SiMAL schema. No explanations, no Markdown, no comments.
- Do NOT include “# ...” comments in the produced schema. Comments are used below only to explain the language.
- Preserve already correct and relevant parts of the existing schema whenever possible.
- Reflect ALL important information from the given files: services, components, configs, APIs, data models, and issues.

RESERVED AND RECOMMENDED KEYWORDS
- Strictly reserved top-level keywords (MUST NOT be used as identifiers for other purposes):
  - system
  - service
  - api
  - endpoints
  - components
  - fields
  - methods

- Commonly used / recommended block kinds and attributes (preferred, but NOT exclusive list):
  - runtime, ci, dependencies, llm_feedback, issues_found
  - struct, database, cache, table, hash, model, job, queue, topic, etc.

You MAY introduce new component kinds or attribute names when needed, as long as the syntax stays valid and the meaning is clear.

------------------------------------------------------------
SiMAL SYNTAX OVERVIEW (with comments explaining the syntax)
DO NOT INCLUDE THESE COMMENTS IN YOUR OUTPUT AS THEY ARE NOT ALLOWED IN SiMAL
------------------------------------------------------------

system {  # Schema starts with exectly one system block. No name, quotes, or colons.
  # Simple key-value attributes. Never drop the colon. Key should be unique within the block.
  # Keys can contain letters, digits, _, ., /, -. but should never be quoted.
  # Attribute values can be single-line strings with or without quotes
  name: Ecommerce Platform  # Optional system name attribute.
  type: microservices  # Free-form attribute: architecture style, domain, etc.
  domain: ecommerce  # Free-form attribute: business / problem domain.
  # Multiline string value using heredoc syntax (<<TEXT ... TEXT).
  description: <<TEXT
This schema defines the ecommerce microservices system context for interaction with LLMs.
It includes services for user management, payment processing, and order handling.
Each service is described with its API endpoints, data models, dependencies,
runtime configurations, and analysis metrics.
TEXT  # End of multiline string.
  # Attribute values can also be maps / dictionaries using { key: value, ... } syntax.
  runtime: {  # System-level runtime environment description.
    # Nested maps are allowed.
    production: {
      description: <<TEXT  # Free-form details about production infra.
AWS EKS cluster in us-west-2 region, 4–8 nodes, autoscaling enabled.
AWS Secrets Manager, Prometheus and Grafana for monitoring.
TEXT
    }
    development: {  # In case deployment info is well-defined, you can use maps as well
      backend: local minikube  # Local / dev environment description.
      nodes: 1
      secrets: local vault
    }
  }

  # Annotations (element metadata) start with @ and are placed before the element they describe. Annotations could be user-specified or LLM-generated.
  # You can only apply annotations to blocks or complex list items (maps or blocks), not to simple attributes.
  # You can even apply multiple annotations to the same element. For instance, two annotations below are applied to the service block.
  @PATH(github.com/org/ecommerce/user-service/)  # Annotation specifying code path for the user service.
  @CALLS(system.verification_service)  # Annotation specifying that this service calls another service. In case of multiple calls, specify multiple @CALLS annotations.
  # In addition to system-level attributes, the system can contain one or more service { ... } blocks.
  # Each service block starts with the service keyword followed by the service name in snake_case (even if the actual service name contains hyphens or camelCase).
  # Service name must be unique within the system and generally each service matches a separate physical service/microservice.
  # However, you can also use multiple service blocks to describe different aspects of the same physical service if needed.
  # For example, if the system is monolithic, you can use multiple service blocks to describe different components of the monolith (e.g., backend and frontend).
  # No matter what, the system must contain at least one service block. Even a library can be described as a service.
  service user_service {  # Service definition. Name = user_service.

    description: Handles user registration, authentication, and profile management.
    langs: [go, protobuf, yaml]                     # Implementation languages. Langs is not a reserved keyword, you can use lang keyword with a string value as well. Note that list elements are comma-separated.
    repos: [github.com/org/ecommerce/user-service,
            github.com/org/ecommerce/user-service-deployment]  # One or more code repositories for this service. Same as langs, repos is not reserved. Could be list of strings or a single string.
    # In addition to simple attributes, service block can contain complex attributes. The list of attributes provided for this user service is not exhaustive.
    # You can introduce new attribute names as needed.
    @PATH(github.com/org/ecommerce/user-service-deployment/deployment/)
    runtime: {  # First complex attribute of service: runtime configurations. Can contain multiple named configs (e.g., development, stage, production-a, production-b, etc.).
      @PATH(github.com/org/ecommerce/user-service-deployment/deployment/dev/)
      development: {
        backend: k8s
        namespace: ecommerce
        replicas: 1
        autoscaling: disabled
        cpu_request: 500m
        memory_request: 256Mi
        cpu_limit: 1
        memory_limit: 512Mi
        deployment_name: user-service
        ports_exposed: [50051]
      }

      @PATH(github.com/org/ecommerce/user-service-deployment/deployment/prod/)
      @BASE(system.user_service.runtime.development)  # Inheritance annotation: production config inherits keys and values from development config unless overridden.
      @DO_NOT_EDIT  # Annotation mostly specified by human to prevent accidental edits by LLMs.
      production: {
        replicas: 3-10
        autoscaling: cpu_target=70%
        cpu_request: 1000m
        memory_request: 512Mi
        cpu_limit: 2000m
        memory_limit: 1024Mi
      }
      # You should follow the same pattern for runtime/deployment config definition and runtime keyword should be preserved.
      # However, you can introduce new keys/attributes as needed to accurately describe the runtime environment if specified in the input files.
    }

    @PATH(github.com/org/ecommerce/user-service-deployment/.circleci/config.yml)
    ci: {  # Second complex attribute of service: CI/CD pipeline description.
      backend: circleci
      steps: {
        CheckoutCode: pull code from repo
        SetupGo: setup Go 1.21 environment
        InstallDependencies: install project dependencies
        RunTests: execute unit and integration tests
        RunLinters: run code linters for quality checks
        BuildBinary: compile the Go binary
        BuildDockerImage: create Docker image
        PushDockerImage: push Docker image to AWS ECR
      }
      # In general, backend and steps should be specified for each ci block, but you can add more keys/attributes as needed.
    }

    # Specifies high-level logical dependencies used by the service.
    # Should not include low-level dependencies like package versions. However, they can be included in case they are critical to the service function.
    dependencies: {
      postgres: main database
      redis: session cache
      verification_service: grpc for email verification
    }

    # Reserved api keyword specifying the service API endpoints. The structure and format of api list must be preserved.
    api: [
      # Each item in the api list is a map describing an API type (e.g., http, grpc, graphql, etc.) and its endpoints.
      # You can introduce new API types (in addition to grpc and http) as needed but they must be specified as maps with type and endpoints keys.
      {
        type: grpc
        endpoints: [  # Each grpc endpoint is specified in a compact format: MethodName(RequestType{arguments}) -> (ResponseType{arguments}) [optional metadata attributes without quotes]
          # input and output types can be tuples or single basic types (str, int, bool, etc.) or complex types (User, CreateUserRequest, etc.)
          # always use colon to separate keys and values in arguments lists
          # never use actual values in request/response types, only types and argument names (e.g., text: "Hello World" or  view: index.html are invalid, use text/view: str instead)
          # if the input or output arg is constant or static view, you can specify basic or complex type and explain it static nature in metadata attributes
          GetUser(GetUserRequest{uuid: str}) -> (user: User{name: str, email: str, verified: bool}?, error: str?) [auth: user_or_admin, cache: 5m, idempotent: true],
          CreateUser(CreateUserRequest{name: str, email: str, password: str}) -> (uuid: str?, error: str?) [auth: none, rate_limit: 10/m, timeout: 5s]
          # ? indicates optional fields in request/response types.
        ]
      },
      # Endpoint groups also support annotations
      @DELETED(timestamp: 2024-05-01T12:00:00Z, reason: "Migrated to gRPC API")  # Annotation indicating the element it annotates is deleted.
      {
        type: http
        endpoints: [
          # Each HTTP endpoint is specified in a compact format: METHOD /path/{with}/{request}/{args} {request args could be specified separately} -> {ResponseType and args} [optional metadata attributes without quotes]
          GET /users/{id} -> JSON{user: User{name: str, email: str, verified: bool}?, error: str?} [auth: user_or_admin, cache: 5m, idempotent: true],
          POST /users JSON{name: str, email: str, password: str} -> JSON{uuid: str?, error: str?} [auth: none, rate_limit: 10/m]
        ]
      }
      # Types other than grpc and http can be specified similarly. Note, the format of endpoint definitions must be preserved. For example:
      # For GraphQL:
      {
        type: graphql
        endpoints: [
          Query.getUser(GetUserInput{id: ID}) -> (user: User{name: str, email: str}?, error: str?) [auth: user_or_admin],
          Mutation.createUser(CreateUserInput{name: str, email: str, password: str}) -> (user: User?, error: str?) [auth: admin],
          Subscription.userUpdated(UserUpdatedFilter{id: ID}) -> (event: UserUpdatedEvent{user: User}) [channel: websocket]
        ]
      }
      # For queue/topic-based APIs:
      {
        type: event
        endpoints: [
          PublishUserCreated(UserCreatedEvent{user_id: str, email: str}) -> Ack{success: bool, error: str?} [topic: users.created, delivery: at_least_once],
          HandleUserDeleted(UserDeletedEvent{user_id: str}) -> void [topic: users.deleted, consumer_group: user-cleanup-workers]
        ]
      }
      # For websocket-based APIs:
      {
        type: websocket
        endpoints: [
          ConnectChat(ChatConnectRequest{user_id: str, room_id: str}) -> Stream{messages: ChatMessage} [auth: bearer, bidirectional: true],
          SubscribeNotifications(NotifySubscribeRequest{user_id: str}) -> Stream{event: NotificationEvent} [auth: bearer, direction: server_to_client]
        ]
      }
    ]

    # Components is another reserved keyword specifying a list of internal service components like databases, caches, jobs, views, structs/classes, etc.
    components: [
      # Component elements can also have annotations. Ideally, each component should have a @PATH annotation specifying the code path where it is implemented/configured.
      @PATH(github.com/org/ecommerce/user-service/internal/db/)
      @CRITICAL  # Human-specified annotation indicating the component is critical for service operation.
      # Each component is a block starting with its kind (e.g., database, cache, struct, etc.) followed by its name in PascalCase.
      # Component kinds are not reserved keywords, you can introduce new kinds as needed. However, you should only specify two words: kind and component name.
      database UserRepo {
        # Inside the component block, you can specify simple attributes
        engine: postgres-12
        description: Manages user data in Postgres. Managed via migrations.
        # and complex attributes like components or lists
        components: [
          # with nested components you can describe internal structure of the component (e.g., tables in a database, fields/methods in a struct, etc.)
          table users {
            id: UUID (PK)
            name: VARCHAR(100)
            email: VARCHAR(100) (UNIQUE)
            password_hash: VARCHAR(256)
            verified: BOOLEAN
            created_at: TIMESTAMP
            updated_at: TIMESTAMP
          }
        ]
        queries: [GetByID, Insert, Update, Delete]
      }

      @PATH(github.com/org/ecommerce/user-service/internal/cache/)
      cache SessionCache {  # Cache component (e.g. Redis).
        engine: redis-6
        description: Caches user session data.

        components: [
          hash sessions {
            session_id: VARCHAR(128) hash key  # type is not strict, you can describe it with plain text
            user_id: UUID
            data: JSONB with personal info and preferences
            expires_at: TIMESTAMP
          }
        ]
        operations: [Set, Get, Delete]
      }

      @PATH(github.com/org/ecommerce/user-service/internal/service/userService.go)
      struct UserService {  # Application struct/class or other entity depending on language.
        # Reserved keyword fields lists the struct fields with optional visibility (+ public, - private, # protected)
        fields: [
          # The only supported format for fields elements is [visibility]field_name: field_type
          -database: UserRepo  # Private database field of type UserRepo.
          -cache: SessionCache
          +Verification: VerificationService
        ]
        # In addition to fields, struct/class can also have methods list specified with reserved methods keyword.
        methods: [
          # Each method is specified in a compact format: [visibility]MethodName(arg1 type, arg2 type, ...) -> returnType
          # ReturnType could be a tuple (user User, err error) or a single type like User.
          # Methods list follows Golang-like function signatures with types after argument names and named return values.
          # However, in contrast to api endpoint definitions, types can't be nested (e.g., User{name: str, email: str} is not allowed here).
          # Fortunately, you can define complex types as separate components list elements and reference them here.
          +GetUser(uuid string) -> *User {  # Public method returning a pointer to User struct.
            # You can also provide method description and other metadata as nested attributes.
            description: Retrieves user by UUID.
          }
          +UpdateUser(uuid string, user User) -> (User, error) {
            description: Updates user details excluding password.
          }
          +CreateUser(name, email, password string) -> (user User, err error) {
            description: Registers new user and sends verification email.
            # For complex methods, you can also provide algorithm description using algo attribute with heredoc syntax.
            algo: <<TEXT
1. Validate basic fields and password strength with validateUserInput
2. Hash password with sha256
3. Insert into users table
4. Call VerifyUser to send verification email
TEXT
            # In addition to description and algo, structs and methods can also have analysis attribute
            # specifying analysis results from various tools (e.g., linters, security scanners, static analysis, etc.)
            # or even LLM-generated feedback.
            analysis: {
              security: [PASSWORD_COMPLEXITY_WEAK]
              linter: [UNHANDLED_ERROR]
            }
            # Feel free to add other simple attributes as needed.
          }
          +VerifyUser(email string) -> bool {
            description: Sends verification email to user.
            algo: <<TEXT
1. Generate verification token
2. Store token in verification service
3. Send verification email via verification service
4. Outside of this flow, user clicks link to verify and verification service triggers user's verified status update
TEXT
            analysis: {
              security: [TOKEN_INSECURE_GENERATION]
            }
          }
          -validateUserInput(name, email, password string) -> bool {
            description: <<TEXT
Validates user input fields:
- Name: non-empty, max 100 chars
- Email: valid format
- Password: minimum 8 chars, includes letters and numbers
TEXT
          }
        ]
      }

      @PATH(github.com/org/ecommerce/user-service/internal/models/user.go)
      struct User {  # Definition of User class used in methods.
        fields: [
          +ID: UUID
          +Name: string
          +Email: string
          -PasswordHash: string
          +Verified: bool
          +CreatedAt: timestamp
          +UpdatedAt: timestamp
        ]
      }
    ]

    llm_feedback: {  # Aggregated feedback from LLM review and tools if available.
      last_review: 2025-11-20T15:30:00Z
      latest_reviewed_commits: {
        github.com/org/ecommerce/user-service: a1b2c3d4e5f6g7h8i9j0k
        github.com/org/ecommerce/user-service-deployment: z9y8x7w6v5u4t3s2r1q0p
      }
      notes: <<TEXT
Improve password hashing to use Argon2id instead of SHA256.
Add rate limiting to CreateUser endpoint to prevent abuse.
TEXT
      issues_found: [
        {
          id: ISSUE-2025-001
          type: security
          description: Password hashing uses SHA256 which is not secure enough.
          recommendation: Migrate to Argon2id for password hashing.
        },
        @IGNORE(reason: "Low severity, to be addressed in future iteration")
        {
          id: ISSUE-2025-002
          type: performance
          description: GetUser endpoint has unoptimized database queries leading to high latency.
          recommendation: Add indexing on frequently queried fields and optimize SQL queries.
        }
      ]
    }
  }

  # Additional services (e.g., verification_service) can be described in the same style.
}

------------------------------------------------------------
ALGORITHM FOR YOUR ACTIONS
------------------------------------------------------------

1. Read the existing schema (if any), the tool outputs, and all <FILE>...</FILE> blocks.
2. Identify services, runtime and ci/cd configs, high-level dependencies, APIs, models, databases, caches, structs, classes, functions, and other important information that appear in the input files.
3. If there is no schema:
   - Create a new system { ... } with appropriate elements.
4. If there is an existing schema:
   - Update it to reflect new or changed information:
     - Add new services/components/methods/APIs when they appear in code.
     - Update existing entries when their implementation or configuration changed.
     - Remove or mark as DELETED elements that are clearly removed from the codebase.
       Note, however, that the abscence of an element in the current files does not imply deletion.
       In most cases, deletion makes sense only when moving to a higher abstraction level (e.g., replacing multiple low-level components with a single higher-level component preserving all key attributes)
       or lowering the abstraction level (e.g., splitting a monolithic service/class into multiple smaller services/classes with or without inheritance).
5. In your schema, focus on capturing important and relevant information. Do NOT include trivial or low-level details unless they are critical to understanding the system.
6. Do NOT forget to add @PATH to all components, services, and other elements where applicable.
7. Also add @CALLS or @BASE annotations when relationships or inheritance are identified.
8. Encode all information in valid SiMAL syntax, following the example and rules above. Make sure the last element is not truncated and the schema is properly enclosed.
9. Remember, SiMAL is not a fork of JSON. You should never use quotes around keys, use colons and commas when needed, and quote values outside of methods/endpoint definitions only when necessary (for example, strings with special symbols).
10. Never place apostrophe, single quote (' or `), quote ("), brackets ([] or {}) inside inline values like description (e.g., retrieve user's id - invalid because of '). If you really need these chars inside value, wrap the whole value into a heredoc <<TEXT ... TEXT instead.
11. Never use JSON-style inline maps in SiMAL. Maps must always be multi-line blocks with proper indentation.
   - Forbidden examples:
     channels: [{ name: stack, driver: stack, channels: [single] }]
     channels: [ { name: stack, driver: stack } ]
     logging Logging { default: stack, channels: [{ name: stack }] }
   - Correct example:
      channels: [
        {
          name: stack
          driver: stack
          channels: [single]
        }
      ]
12. Do NOT add empty values when information is missing.
13. Do NOT provide LLM feedback or analysis unless explicitly given in the input.
14. Do NOT output any comments, explanations, or Markdown. Only the final SiMAL schema.
15. Do NOT hallucinate information that is not present in the input files or existing schema.
16. Do NOT copy-paste any content from sample schema with explanations provided above.

Now read the inputs and produce the final SiMAL schema.""".strip()


def build_simal_prompt_text(
    existing_schema: Optional[str],
    tool_analysis_for_chunk: str,
    chunk_files: List[Tuple[Path, str]],
    root: Path,
) -> str:
    existing_schema_str = (existing_schema or "").strip()
    tool_analysis_str = (tool_analysis_for_chunk or "").strip()
    file_blocks = build_file_blocks(chunk_files, root)
    files_section = "\n\n".join(file_blocks)
    simal_instructions = get_simal_instructions()

    full_prompt = (
        f"{simal_instructions}\n\n"
        "<<EXISTING_SCHEMA>>\n"
        f"{existing_schema_str}\n"
        "<<END_EXISTING_SCHEMA>>\n\n"
        "<<TOOL_ANALYSIS>>\n"
        f"{tool_analysis_str}\n"
        "<<END_TOOL_ANALYSIS>>\n\n"
        "<<FILES>>\n"
        f"{files_section}\n"
        "<<END_FILES>>"
    )
    return full_prompt


def build_simal_messages(
    existing_schema: Optional[str],
    tool_analysis_for_chunk: str,
    chunk_files: List[Tuple[Path, str]],
    root: Path,
) -> List[Dict[str, str]]:
    user_content = build_simal_prompt_text(
        existing_schema=existing_schema,
        tool_analysis_for_chunk=tool_analysis_for_chunk,
        chunk_files=chunk_files,
        root=root,
    )

    system_msg = {
        "role": "system",
        "content": "You are a precise assistant that ONLY outputs valid SiMAL schemas, with no explanations.",
    }

    return [
        system_msg,
        {"role": "user", "content": user_content},
    ]


# ---------- Together.ai call ----------

from requests.exceptions import (
    ChunkedEncodingError,
    ConnectionError,
    Timeout,
    RequestException,
)
import time

def call_together_update_schema(
    schema_so_far: Optional[str],
    chunk_files: List[Tuple[Path, str]],
    root: Path,
    model: str,
    api_key: str,
    log_input_path: str,
    tool_analysis_struct: Optional[Dict],
    referer: Optional[str] = None,
    app_title: Optional[str] = None,
    reasoning_effort: str = "medium",
    temperature: float = 0.1,
    max_retries=3,
    retry_backoff_sec: float = 10.0,
) -> str:
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    if referer:
        headers["HTTP-Referer"] = referer
    if app_title:
        headers["X-Title"] = app_title

    tool_analysis_for_chunk = build_tool_analysis_for_chunk(
        chunk_files=chunk_files,
        root=root,
        tool_analysis_struct=tool_analysis_struct,
    )
    messages = build_simal_messages(
        existing_schema=schema_so_far,
        tool_analysis_for_chunk=tool_analysis_for_chunk,
        chunk_files=chunk_files,
        root=root,
    )

    with open(log_input_path, "w") as fw:
      json.dump(messages, fw)

    body: Dict = {
        "model": model,
        "messages": messages,
        "stream": False,
        "temperature": float(temperature),
        "reasoning_effort": reasoning_effort,
    }

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                TOGETHER_URL,
                headers=headers,
                data=json.dumps(body),
                timeout=600,
            )
            response.raise_for_status()
            data = response.json()
            content = data["choices"][0]["message"]["content"]
            return content

        except (ChunkedEncodingError, ConnectionError, Timeout) as e:
            # transient errors: log + retry
            last_error = e
            print(
                f"[WARN] Together.ai transient error on attempt {attempt}/{max_retries}: {e}",
                file=sys.stderr,
            )
            if attempt < max_retries:
                time.sleep(retry_backoff_sec * attempt)  # simple backoff
            continue

        except RequestException as e:
            # Other HTTP errors -> don’t aggressively retry
            last_error = e
            print(
                f"[ERROR] Together.ai HTTP error on attempt {attempt}/{max_retries}: {e}",
                file=sys.stderr,
            )
            break

        except (KeyError, IndexError, ValueError) as e:
            last_error = e
            print(
                f"[ERROR] Unexpected Together.ai response format: {e}",
                file=sys.stderr,
            )
            break

    # If we get here, all retries failed
    raise RuntimeError(f"Together.ai call failed after {max_retries} attempts: {last_error}")


# ---------- OpenAI call (no splitting, char limit enforced in chunking) ----------

OPENAI_URL = "https://api.openai.com/v1/chat/completions"

def call_openai_update_schema(
    schema_so_far: Optional[str],
    chunk_files: List[Tuple[Path, str]],
    root: Path,
    model: str,
    api_key: str,
    log_input_path: str,
    tool_analysis_struct: Optional[Dict],
    reasoning_effort: str = "medium",
    temperature: float = 0.1,
    max_retries: int = 3,
    retry_backoff_sec: float = 10.0,
) -> str:
    """
    Same behavior as call_together_update_schema, but using OpenAI's
    /v1/chat/completions endpoint instead of Together.ai.

    Note:
      - `model` should be an OpenAI chat model id.
      - `api_key` is the OpenAI API key (or compatible).
    """
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    tool_analysis_for_chunk = build_tool_analysis_for_chunk(
        chunk_files=chunk_files,
        root=root,
        tool_analysis_struct=tool_analysis_struct,
    )
    messages = build_simal_messages(
        existing_schema=schema_so_far,
        tool_analysis_for_chunk=tool_analysis_for_chunk,
        chunk_files=chunk_files,
        root=root,
    )

    # Log raw messages for debugging
    if log_input_path:
        try:
            with open(log_input_path, "w", encoding="utf-8") as fw:
                json.dump(messages, fw, ensure_ascii=False, indent=2)
        except OSError as e:
            print(f"[WARN] Failed to log OpenAI input to {log_input_path}: {e}", file=sys.stderr)

    body: Dict = {
        "model": model,
        "messages": messages,
        #"temperature": float(temperature),
        # reasoning_effort is only supported on some models; harmless if ignored
        "reasoning_effort": reasoning_effort,
        "response_format": {
          "type": "text"
        },
    }

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                OPENAI_URL,
                headers=headers,
                data=json.dumps(body),
                timeout=600 * 3,
            )
            response.raise_for_status()
            data = response.json()
            content = data["choices"][0]["message"]["content"]
            return content

        except (ChunkedEncodingError, ConnectionError, Timeout) as e:
            last_error = e
            print(
                f"[WARN] OpenAI transient error on attempt {attempt}/{max_retries}: {e}",
                file=sys.stderr,
            )
            if attempt < max_retries:
                time.sleep(retry_backoff_sec * attempt)
            continue

        except RequestException as e:
            last_error = e
            print(
                f"[ERROR] OpenAI HTTP error on attempt {attempt}/{max_retries}: {e}",
                file=sys.stderr,
            )
            #break
            if attempt < max_retries:
                time.sleep(retry_backoff_sec * attempt)
            continue

        except (KeyError, IndexError, ValueError) as e:
            last_error = e
            print(
                f"[ERROR] Unexpected OpenAI response format: {e}",
                file=sys.stderr,
            )
            break

    raise RuntimeError(f"OpenAI call failed after {max_retries} attempts: {last_error}")


In [ ]:
# ---------- Main orchestration ----------

def generate_repo_schema(
    repo_url: Optional[str] = None,
    repo_path: Optional[str] = None,
    dest: str = "repo",
    model: str = DEFAULT_MODEL,
    max_lines_per_chunk: int = 1000,
    max_file_size_mb: float = 2.0,
    schema_output: str = "simal_schema.txt",
    tool_analysis_path: Optional[str] = None,
    referer: Optional[str] = None,
    app_title: str = "simal-schema-builder",
    reasoning_effort: str = "medium",
    temperature: float = 0.1,
    max_input_chars: int = 200_000,
    start_step: int = 0,
) -> str:
    api_key = os.getenv("API_KEY")
    if not api_key:
        print("ERROR: API_KEY environment variable is not set.", file=sys.stderr)
        sys.exit(1)

    # Determine repo root
    if repo_path:
        root = Path(repo_path).resolve()
    else:
        if not repo_url:
            print("ERROR: Either --repo-path or --repo-url must be provided.", file=sys.stderr)
            sys.exit(1)
        dest_dir = Path(dest).resolve()
        run_git_clone(repo_url, dest_dir)
        root = dest_dir

    if not root.exists():
        print(f"ERROR: Repo path {root} does not exist.", file=sys.stderr)
        sys.exit(1)

    # Prepare schema output naming (step files)
    schema_base_path = Path(schema_output).resolve()
    base_stem = schema_base_path.stem
    ext = schema_base_path.suffix

    def step_path(step: int, ext: str = ext) -> Path:
        return schema_base_path.with_name(f"{base_stem}_step={step}{ext}")

    # Try loading existing schema from step=0 if present, otherwise from plain file
    step = 0
    schema_so_far: Optional[str] = None
    initial_step_path = step_path(step)

    if initial_step_path.exists():
        print(f"[INFO] Loading existing schema from {initial_step_path}", file=sys.stderr)
        try:
            schema_so_far = initial_step_path.read_text(encoding="utf-8")
        except OSError as e:
            print(f"[WARN] Cannot read existing schema file: {e}", file=sys.stderr)
    elif schema_base_path.exists():
        print(f"[INFO] Loading existing schema from {schema_base_path}", file=sys.stderr)
        try:
            schema_so_far = schema_base_path.read_text(encoding="utf-8")
        except OSError as e:
            print(f"[WARN] Cannot read existing schema file: {e}", file=sys.stderr)

    # Load tool analysis once, if provided
    tool_analysis_struct: Optional[Dict] = None
    if tool_analysis_path:
        tpath = Path(tool_analysis_path).resolve()
        tool_analysis_struct = load_tool_analysis_struct(tpath)
        if tool_analysis_struct:
            print(f"[INFO] Loaded tool analysis from {tpath}", file=sys.stderr)

    processed_files: Set[Path] = set()
    max_file_size_bytes = int(max_file_size_mb * 1024 * 1024)
    max_lines = max_lines_per_chunk

    print(f"[INFO] Starting BFS traversal from {root}", file=sys.stderr)

    for directory in bfs_directories(root):
        rel_dir = directory.relative_to(root)
        print(f"[INFO] Processing directory: {rel_dir}", file=sys.stderr)

        files = collect_files_in_dir(directory)

        for chunk_files in build_chunks_for_dir(
            files,
            root=root,
            processed_files=processed_files,
            max_lines_per_chunk=max_lines,
            max_file_size_bytes=max_file_size_bytes,
            max_chars_per_chunk=max_input_chars,
        ):
            if start_step > 0 and step < start_step:
                print(f"[INFO] Skipping chunk at step {step} due to start_step={start_step}", file=sys.stderr)
                step += 1
                continue
            elif start_step > 0 and step == start_step:
                schema_so_far_path = step_path(step)
                print(f"[INFO] Resuming from step {step} at {schema_so_far_path}", file=sys.stderr)
                schema_so_far = schema_so_far_path.read_text(encoding="utf-8")

            chunk_paths = [p for p, _ in chunk_files]
            print(
                f"[INFO] Sending chunk with {len(chunk_paths)} file(s): "
                + ", ".join(str(p.relative_to(root)) for p in chunk_paths),
                file=sys.stderr,
            )

            try:
                schema_so_far = call_openai_update_schema(
                    schema_so_far=schema_so_far,
                    chunk_files=chunk_files,
                    root=root,
                    model=model,
                    api_key=api_key,
                    log_input_path=step_path(step+1, ext=".input_log"),
                    tool_analysis_struct=tool_analysis_struct,
                    #referer=referer,
                    #app_title=app_title,
                    reasoning_effort=reasoning_effort,
                    temperature=temperature,
                )
            except Exception as e:
                print(f"[ERROR] API call failed: {e}", file=sys.stderr)
                raise e

            # Persist schema after each step
            step += 1
            schema_output_path = step_path(step)
            try:
                schema_output_path.write_text(schema_so_far or "", encoding="utf-8")
            except OSError as e:
                print(f"[WARN] Failed to write schema to {schema_output_path}: {e}", file=sys.stderr)

    final_path = step_path(step) if step > 0 else schema_base_path

    print(f"[INFO] Finished. Final SiMAL schema written to {final_path}", file=sys.stderr)
    return final_path


from pathlib import Path
from typing import List, Optional


def batch_generate_repo_schemas(
    repo_urls: List[str],
    base_output_dir: str,
    model: str = "gpt-5-2025-08-07",
    reasoning_effort: str = "medium",
    max_lines_per_chunk: int = 5000,
    max_input_chars: int = 250_000,
    temperature: float = 0.1,
    tool_analysis_path: Optional[str] = None,
    referer: Optional[str] = None,
    app_title: str = "simal-schema-builder",
    start_step: int = 0,
) -> None:
    """
    For each repo URL:
      - Create a separate folder under base_output_dir.
      - Clone the repo into <repo_folder>/repo
      - Write that repo's SiMAL schema to <repo_folder>/schema_output.txt
      - Call generate_repo_schema(...) with the given parameters.
    """

    final_path_per_repo = {}

    base_dir = Path(base_output_dir).resolve()
    base_dir.mkdir(parents=True, exist_ok=True)

    for repo_url in repo_urls:
        # Derive a safe folder name from repo URL
        slug = repo_url.rstrip("/").split("/")[-1]
        if slug.endswith(".git"):
            slug = slug[:-4]

        repo_folder = base_dir / slug
        repo_folder.mkdir(parents=True, exist_ok=True)

        # Where to clone this repo
        dest_dir = repo_folder / "repo"

        # Where to save the schema for this repo
        simal_schema_folder = repo_folder / "simal_schemas"
        simal_schema_folder.mkdir(parents=True, exist_ok=True)
        schema_output_path = simal_schema_folder / "schema_output.txt"

        print(f"[BATCH] Processing {repo_url}")
        print(f"[BATCH]  -> dest: {dest_dir}")
        print(f"[BATCH]  -> schema_output: {schema_output_path}")

        final_schema_path = generate_repo_schema(
            repo_url=repo_url,
            dest=str(dest_dir),
            reasoning_effort=reasoning_effort,
            model=model,
            schema_output=str(schema_output_path),
            max_lines_per_chunk=max_lines_per_chunk,
            max_input_chars=max_input_chars,
            temperature=temperature,
            tool_analysis_path=tool_analysis_path,
            referer=referer,
            app_title=app_title,
            start_step=start_step,
        )
        final_path_per_repo[repo_url] = final_schema_path



In [ ]:
os.environ["API_KEY"] = "<YOUR_API_KEY>"

In [ ]:
mkdir schemas

In [ ]:
repo_urls = [
    "https://github.com/huggingface/tokenizers",  # Python ML library

    #"https://github.com/elgris/microservice-app-example",  # PHP

    #"https://github.com/luisotavio756/dashboard-reactjs",

    #"https://github.com/dgildeh/otel-python-cloud-run",

    #"https://github.com/mehdihadeli/spring-food-delivery-microservices"

    #"https://github.com/ThreeDotsLabs/wild-workouts-go-ddd-example"
]

batch_generate_repo_schemas(
    repo_urls=repo_urls,
    base_output_dir="/content/schemas",
    reasoning_effort="medium",
    model="gpt-5-2025-08-07",
    max_lines_per_chunk=5000,
    max_input_chars=250_000,
    temperature=0.1,
    start_step=22
)

In [ ]:
[
    "https://github.com/fastapi/full-stack-fastapi-template",  # FastAPI + React + SQLModel + Postgres monorepo, with Docker & GitHub Actions

    "https://github.com/dgildeh/otel-python-cloud-run",  # Two Python microservices on Google Cloud Run, instrumented with OpenTelemetry; good for tracing/infra structure

    "https://github.com/fastapi/sqlmodel",  # SQLModel library (ORM-ish on top of SQLAlchemy + Pydantic), heavily used in FastAPI stacks

    "https://github.com/ThreeDotsLabs/wild-workouts-go-ddd-example",  # Go DDD example app with hexagonal-ish architecture, CQRS, and front-end client

    "https://github.com/mehdihadeli/spring-food-delivery-microservices",  # Food-delivery microservices in Java + Spring Boot, DDD, CQRS, vertical slices

    "https://github.com/luisotavio756/dashboard-reactjs",  # Simple React dashboard app, MIT-licensed; great as a small front-end-only corpus

    "https://github.com/elgris/microservice-app-example",  # Example microservice app with several components written in different languages; still small enough for a single context and great for polyglot schema evaluation,

    "https://github.com/niksyromyatnikov/JuniorTest",  # PHP

    "https://github.com/niksyromyatnikov/OHLCFormer",  # Python ML library

    "https://github.com/huggingface/tokenizers",  # Rust tokenizer ML

]